# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR^2 dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset metadata
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
List record sets and their fields by their `@id`s.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print("  Fields (by @id):")
    for field in rs.get('field', []):
        if isinstance(field, dict):
            field_id = field.get('@id', 'N/A')
            field_name = field.get('name', 'N/A')
        else:  # it might be a plain @id string
            field_id = field
            field_name = 'N/A'
        print(f"    - {field_id} (name: {field_name})")
    print()

## 3. Data Extraction
Load data from the main record set(s) into a DataFrame for analysis. Reference record sets and fields by their `@id`.

In [ ]:
# Select record sets of interest
main_record_set_id = None
for rs in record_sets:
    # Heuristically select the first or dataset-like record set
    if 'Dataset' in rs.get('@type', '') or 'dataset' in rs.get('name', '').lower() or main_record_set_id is None:
        main_record_set_id = rs['@id']

if main_record_set_id is None:
    raise ValueError("No record set found.")

all_record_set_ids = [rs['@id'] for rs in record_sets]

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in all_record_set_ids:
    print(f"Loading records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show the columns (fields) of the main record set
print(f"\nColumns in main record set (@id={main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, categorization, and grouping operations. Use field and record set `@id`s.

In [ ]:
# Select numeric and group fields by their @id from DataFrame columns
df = dataframes[main_record_set_id]
print("Available fields:")
for idx, field_name in enumerate(df.columns):
    print(f"  {idx}: {field_name}")

# Heuristically select a likely numeric field, e.g., interval between cancers, or age
candidate_numeric_fields = [col for col in df.columns if 'interval' in col.lower() or 'age' in col.lower()]
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]
else:
    numeric_field_id = df.columns[0]  # fallback

print(f"\nNumeric field selected: {numeric_field_id}")

# Heuristically select a candidate group field (e.g., sex, MSI status, anatomical site)
candidate_group_fields = [col for col in df.columns if any(x in col.lower() for x in ["msi", "anatomical", "sex", "site"]) ]
if candidate_group_fields:
    group_field_id = candidate_group_fields[0]
else:
    group_field_id = df.columns[1]

print(f"Group (categorical) field selected: {group_field_id}\n")
# Filter records with numeric_field > threshold
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()  # use mean as example threshold
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    # Try conversion if possible
    try:
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > 10].copy()
        filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        threshold = 10
    except Exception as e:
        print(f"Could not filter on field {numeric_field_id}: {e}")
        filtered_df = df.copy()
        threshold = None

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = ((filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std())
else:
    # Try conversion again if needed
    filtered_df[f"{numeric_field_id}_normalized"] = ((pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std())

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group and compute mean
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution and relationships in the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], kde=True, bins=10, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot of the numeric field grouped by a categorical field
if group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
- We successfully loaded the FAIR² colorectal cancer survivors dataset with `mlcroissant` via its Croissant schema.
- Main record sets and fields were identified by their `@id`s, with data extracted to Pandas DataFrames.
- Demonstrated basic filtering, normalization, grouping, and visual analytics over one key numeric and one categorical field.
- This approach can be reused for detailed, reproducible exploration and preprocessing of `mlcroissant`-packaged datasets.